# Zagent Model-RSI on Kaggle (GPU)

在 Kaggle 免费 GPU 上跑 alpha 训练配方进化。**步骤：先 Add Data 挂载 csi500-seq-trans 数据集，再开 GPU，然后 Run All。**

In [ ]:
# ===== 1. 安装 zagent（一行）=====
!pip install -q git+https://github.com/Z-Asset/zagent.git
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
print("zagent 已装")

In [ ]:
# ===== 2. 检查 GPU =====
import torch
print("CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("警告：未检测到 GPU，请在右侧 Settings 里开 GPU accelerator")

## 3. 配置
数据集挂载后在 /kaggle/input/ 下。

In [ ]:
PKG = "/kaggle/input/csi500-seq-trans"   # 数据集挂载路径
import os
print("数据包文件:", os.listdir(PKG))

## 4. 体检

In [ ]:
from zagent.migrate import doctor
rc = doctor(PKG)
if rc != 0:
    raise SystemExit("体检未通过")

## 5. Model-RSI 全量 walk-forward（GPU）

GPU 上跑全量 123 块 + 15 epochs，约 30 分钟。

In [ ]:
from zagent.model import run_alpha
from pathlib import Path
# GPU 全量：blocks=None(全量123块)、fixed_epochs=15、多seed降噪
SETTINGS = {
    "iterations": 4,       # TPE 搜索轮数（每轮都训全量，4轮约2小时，可先1轮试）
    "blocks": None,        # 全量 123 块
    "max_train": 200000,
    "fixed_epochs": 15,
    "n_seeds": 1,          # GPU 快可设 2-3 降噪
    "threads": 8,
    "isolated": False,     # Kaggle 环境无需子进程隔离
}
stats = run_alpha(Path(PKG), SETTINGS, offline=True)
print("最佳 RankIC:", round(stats.best_rank_ic, 4))
print("最佳配方:", stats.best_recipe)